<a href="https://colab.research.google.com/github/VincentCCL/MTAT/blob/main/notebooks/MTAT26_COMET.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#3.4.5 COMET


First we need to install some specific packages.

In [1]:
!pip -q install "transformers==4.41.2" "huggingface-hub==0.23.4"
!pip -q install unbabel-comet


##**IMPORTANT: Restart RUNTIME**

And here is the python code for a single sentence. Note that COMET expects a list of sentence-level dictionaries, one dictionary per sentence.
So for multiple sentences, you just have multiple dictionaries in the list.

In [2]:
import torch
from comet import download_model, load_from_checkpoint

ckpt = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(ckpt)

gpus = 1 if torch.cuda.is_available() else 0

data = [
    {"src": "She eats apples.", "mt": "Zij eet appels.", "ref": "Zij eet appels."}
]

out = model.predict(data, batch_size=8, gpus=gpus)
print("\n",out.system_score)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages


 0.9804219007492065


## Command line example

We download the full source file `dev.en` and the full reference file `dev.nl` (1000 sentences each)

In [3]:
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.en
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.nl

--2026-03-05 11:49:51--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.en
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 32441 (32K) [text/plain]
Saving to: ‘dev.en’

dev.en              100%[===================>]  31.68K  --.-KB/s    in 0.001s  

2026-03-05 11:49:51 (21.5 MB/s) - ‘dev.en’ saved [32441/32441]

--2026-03-05 11:49:51--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/dev.nl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 34843 (34K) [te

We use the `head` command to create new files containing only the first 30 lines.

In [4]:
!head -n 30 dev.en > dev30.en
!head -n 30 dev.nl > dev30.nl

Then we download a python script that calculates the COMET score

In [5]:
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/comet_score_files.py

--2026-03-05 11:49:59--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/comet_score_files.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2015 (2.0K) [text/plain]
Saving to: ‘comet_score_files.py’

comet_score_files.p 100%[===================>]   1.97K  --.-KB/s    in 0s      

2026-03-05 11:49:59 (43.4 MB/s) - ‘comet_score_files.py’ saved [2015/2015]



Now we download the bing translations of these 30 lines.

In [6]:
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/tatoeba-bing.txt

--2026-03-05 11:50:03--  https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/data/tatoeba-en-nl/tatoeba-bing.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 989 [text/plain]
Saving to: ‘tatoeba-bing.txt’

tatoeba-bing.txt    100%[===================>]     989  --.-KB/s    in 0s      

2026-03-05 11:50:03 (60.1 MB/s) - ‘tatoeba-bing.txt’ saved [989/989]



And then we run the script with three arguments:
1. the source file `dev30.en`
2. the mt output, which we just downloaded from the web
3. the reference file `dev30.nl`

In [8]:
!python comet_score_files.py dev30.en tatoeba-bing.txt dev30.nl

Fetching 5 files: 100% 5/5 [00:00<00:00, 68759.08it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Encoder model frozen.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging a